# Run this notebook on Google Colab

In [1]:
!pip install optuna --quiet

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import random
import tensorflow as tf
from math import sqrt
import math

# Import Optuna for hyperparameter tuning
import optuna

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from keras.models import Sequential
from keras.layers import GRU, Dropout, Dense
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Ensure TensorFlow is using GPU
device_name = tf.test.gpu_device_name()
if device_name:
    print(f"GPU Detected: {device_name}")
else:
    print("No GPU found. Training may be slow.")

GPU Detected: /device:GPU:0


In [5]:
# Train-val-test split (70% training, 20% validation, 10% test)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.20
TEST_RATIO = 0.10

In [6]:
# Update dataset path for Google Drive
data_path = "/content/drive/My Drive/Colab Notebooks/5.0/Bitcoin_data.csv"
data = pd.read_csv(data_path, index_col=0, parse_dates=True)

data

,Open,High,Low,Close,VolumeBTC,VolumeUSD,stoch_k,stoch_d,momentum,macd,macd_signal,atr_short,atr_long,vol_ratio,adx
date,,,,,,,,,,,,,,,
2024-01-01 00:33:00,42367.0,42373.0,42365,42371,0.016585,702.728120,9.166176,6.600790,-94.0,2.261327,24.403575,19.252096,18.831203,102.235086,47.478318
2024-01-01 00:34:00,42357.0,42357.0,42351,42351,0.019690,833.891190,6.163173,7.564574,-121.0,-3.058470,18.911166,19.326887,18.889643,102.314729,44.942988
2024-01-01 00:35:00,42359.0,42359.0,42351,42359,0.031363,1328.501505,3.935135,6.421494,-117.0,-6.553370,13.818259,18.194198,18.345160,99.177099,42.503637
2024-01-01 00:36:00,42368.0,42377.0,42356,42377,0.034081,1444.249689,9.066667,6.388325,-78.0,-7.780961,9.498415,18.474778,18.477902,99.983092,39.498642
2024-01-01 00:37:00,42395.0,42411.0,42392,42403,0.005832,247.290480,22.933333,11.978378,-33.0,-6.580002,6.282732,20.027300,19.254007,104.016271,37.792666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-10 23:39:00,97347.0,97347.0,97339,97339,0.005864,570.787135,57.060236,51.188618,53.0,-10.792340,-12.042076,28.589994,32.832443,87.078484,13.563130
2025-02-10 23:40:00,97339.0,97339.0,97336,97336,0.016011,1558.430149,61.300897,56.935688,-10.0,-10.456826,-11.725026,26.030994,31.340821,83.057793,12.786939
2025-02-10 23:41:00,97340.0,97346.0,97340,97346,0.001320,128.496720,68.840580,62.400571,46.0,-9.277072,-11.235435,24.427895,30.273780,80.689940,12.304698


In [7]:
print("Missing values:\n", data.isna().sum())

Missing values:
 Open           0
High           0
Low            0
Close          0
VolumeBTC      0
VolumeUSD      0
stoch_k        0
stoch_d        0
momentum       0
macd           0
macd_signal    0
atr_short      0
atr_long       0
vol_ratio      0
adx            0
dtype: int64


In [8]:
# Identify target column index
target_col_idx = data.columns.get_loc('Close')

In [9]:
# Train-val-test split
train_index = int(len(data) * TRAIN_RATIO)
val_index = train_index + int(len(data) * VAL_RATIO)

train_data = data.iloc[:train_index]
val_data = data.iloc[train_index:val_index]
test_data = data.iloc[val_index:]

print("Train shape:", train_data.shape)
print("Validation shape:", val_data.shape)
print("Test shape:", test_data.shape)

Train shape: (395957, 15)
Validation shape: (113130, 15)
Test shape: (56567, 15)


In [10]:
# Fit scaler ONLY on training data
scaler = MinMaxScaler()
scaler.fit(train_data)

# Transform datasets
train_scaled = scaler.transform(train_data)
val_scaled = scaler.transform(val_data)
test_scaled = scaler.transform(test_data)

In [11]:
# Function to create sequences for time series forecasting
def create_sequences(data_array, window_size, target_col_idx):
    X, y = [], []
    for i in range(len(data_array) - window_size):
        X.append(data_array[i:(i + window_size), :])
        y.append(data_array[i + window_size, target_col_idx])
    return np.array(X), np.array(y)

In [12]:
# Function to inverse transform predictions
def inverse_transform_predictions(predicted_scaled, y_actual_scaled, scaler, train_data, target_col_idx):
    predicted_full = np.zeros((predicted_scaled.shape[0], train_data.shape[1]))
    y_actual_full = np.zeros((y_actual_scaled.shape[0], train_data.shape[1]))

    predicted_full[:, target_col_idx] = predicted_scaled[:, 0]
    y_actual_full[:, target_col_idx] = y_actual_scaled

    predicted_inverse = scaler.inverse_transform(predicted_full)[:, target_col_idx]
    y_actual_inverse = scaler.inverse_transform(y_actual_full)[:, target_col_idx]

    return predicted_inverse, y_actual_inverse

In [13]:
# Objective function for Optuna
def objective(trial):
    """
    Objective function for Optuna optimization.

    Returns:
    - Validation RMSE (Root Mean Squared Error).
    """
    # Suggest hyperparameters
    window_size = trial.suggest_int("WINDOW_SIZE", 30, 120, step=10)
    num_neurons = trial.suggest_int("NUM_NEURONS", 32, 128, step=16)
    dropout_rate = trial.suggest_float("DROPOUT_RATE", 0.1, 0.5, step=0.05)
    patience = trial.suggest_int("PATIENCE", 3, 10)
    epochs = trial.suggest_int("EPOCHS", 50, 200, step=10)
    batch_size = trial.suggest_categorical("BATCH_SIZE", [32, 64, 128, 256, 512])
    learning_rate = trial.suggest_loguniform("LEARNING_RATE", 1e-5, 1e-2)

    # Create sequences
    X_train, y_train = create_sequences(train_scaled, window_size, target_col_idx)
    X_val, y_val = create_sequences(val_scaled, window_size, target_col_idx)

    # Build the GRU model
    model = Sequential([
        GRU(num_neurons, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(dropout_rate),
        Dense(1)
    ])

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="mean_squared_error")

    # Early stopping
    early_stopping = EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=0)

    # Train the model with GPU support
    with tf.device('/device:GPU:0'):
        model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(X_val, y_val),
            callbacks=[early_stopping],
            verbose=0
        )

    # Predict on validation set
    y_val_pred_scaled = model.predict(X_val, verbose=0)

    # Inverse transform predictions
    predicted_val_inverse, y_val_inverse = inverse_transform_predictions(
        predicted_scaled=y_val_pred_scaled,
        y_actual_scaled=y_val,
        scaler=scaler,
        train_data=train_data,
        target_col_idx=target_col_idx
    )

    # Compute validation RMSE
    val_rmse = sqrt(mean_squared_error(y_val_inverse, predicted_val_inverse))

    return val_rmse  # Minimize RMSE

In [14]:
# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)  # Run for 20 trials

# Display best parameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

[I 2025-02-15 21:51:44,895] A new study created in memory with name: no-name-a98f3846-f553-421a-a7cf-2f86887aaa7c
[I 2025-02-15 21:55:03,353] Trial 0 finished with value: 1282.723201591954 and parameters: {'WINDOW_SIZE': 110, 'NUM_NEURONS': 112, 'DROPOUT_RATE': 0.1, 'PATIENCE': 6, 'EPOCHS': 90, 'BATCH_SIZE': 256, 'LEARNING_RATE': 0.00013220997642149948}. Best is trial 0 with value: 1282.723201591954.
[I 2025-02-15 21:56:25,581] Trial 1 finished with value: 5455.672529501652 and parameters: {'WINDOW_SIZE': 90, 'NUM_NEURONS': 48, 'DROPOUT_RATE': 0.5, 'PATIENCE': 5, 'EPOCHS': 100, 'BATCH_SIZE': 256, 'LEARNING_RATE': 8.540303538574209e-05}. Best is trial 0 with value: 1282.723201591954.
[I 2025-02-15 22:00:49,590] Trial 2 finished with value: 1782.053244641219 and parameters: {'WINDOW_SIZE': 70, 'NUM_NEURONS': 64, 'DROPOUT_RATE': 0.35, 'PATIENCE': 6, 'EPOCHS': 150, 'BATCH_SIZE': 64, 'LEARNING_RATE': 0.00043824867229914845}. Best is trial 0 with value: 1282.723201591954.
[I 2025-02-15 22:03

Best Hyperparameters: {'WINDOW_SIZE': 100, 'NUM_NEURONS': 96, 'DROPOUT_RATE': 0.15000000000000002, 'PATIENCE': 9, 'EPOCHS': 160, 'BATCH_SIZE': 128, 'LEARNING_RATE': 0.003435926305551255}
